# Professional Activities

In [ ]:
#| default_exp ProfessionalActivities

In [ ]:
#| export
#| hide
from biobib.table import Table, tex_escape, colonify, to_int
from biobib.doc import Doc, NumbersDoc
from nbdev import show_doc
import pandas as pd
import re

/Users/kellycaylor/mambaforge/envs/biobib/lib/python3.10/site-packages/numbers_parser/__init__.py:61: UserWarning: Numbers version 14.1 not tested with this version
  warnings.warn(f"Numbers version {installed_version} not tested with this version")


In [ ]:
#| hide
#| export
from fastcore.test import *
from fastcore.test import *
from fastcore.basics import *
from fastcore.foundation import *

### Define a template for professional activities

Here is a template we can use for professional activities. It generates a longtable which can span multiple pages, and provides the year, role, and type of each service activity.


In [ ]:
#| export
professional_activities_template = r'''% UC Bio-bib Professional Generic Activity Table
% Created on \VAR{created}

\BLOCK{for activity in items}
\VAR{activity.Year|to_int} & \VAR{activity.Role|tex_escape} & \VAR{activity.Organization|tex_escape} \\
\BLOCK{endfor}

'''

In [ ]:
#| export 
class ProfessionalActivities(Table):
    """
    A Professional Activities table for Biobibs and CVs
    """
    def __init__( 
            self,
            doc:Doc=None, # Google sheet object
            worksheet:str='Additional Professional Activites',   # worksheet name in google sheet
            csv_file:str=None, # optional csv file (if using csv files) 
            table_name:str='ProfessionalActivities', # Will be determined from category.
            cumulative:bool=False, # Is this table cumulative? 
            ascending:bool=True, # Sort in ascending order?
            template=professional_activities_template # template content
    ):
        super().__init__(
            doc=doc,
            worksheet=worksheet, 
            table_name=table_name,
            csv_file=csv_file, 
            template=template)
        self.ascending = ascending
        self.cumulative = cumulative
        self.df = self.clean_df()
        print(f'Initialized {self.table_name} table')

    def clean_df(self):
        df = super().table_clean_df()
        df = df.sort_values(by=['Year', 'Role'], ascending=[True, False])
        return df


class SpecialAppointments(Table):
    """
    A Special Appointments table for biobibs and CVs
    """
    def __init__(            
            self,
            doc:Doc=None, # Google sheet object
            worksheet:str='Special Appointments',   # worksheet name in google sheet
            csv_file:str=None, # optional csv file (if using csv files) 
            table_name:str='SpecialAppointments', # Will be determined from category.
            cumulative:bool=False, # Is this table cumulative? 
            ascending:bool=True, # Sort in ascending order?
            template=professional_activities_template # template content
        ):
        super().__init__(
            doc=doc,
            worksheet=worksheet, 
            table_name=table_name,
            csv_file=csv_file, 
            template=template)
        self.ascending = ascending
        self.cumulative = cumulative
        self.df = self.clean_df()
        print(f'Initialized {self.table_name} table')

    def clean_df(self):
        df = super().table_clean_df()
        df = df.sort_values(by=['Year', 'Role'], ascending=[True, False])
        return df


In [ ]:
sheet_name = "CV.numbers"
doc = NumbersDoc(sheet_name)
doc.get_dataframe

<bound method NumbersDoc.get_dataframe of <biobib.doc.NumbersDoc object>>

In [ ]:
table1 = ProfessionalActivities(
    doc=doc,worksheet='Additional Professional Activities',
    table_name='Professional Activities',
    cumulative=True,
    ascending=False,
    template=professional_activities_template
)

Cleaning table for  Professional Activities
Initialized Professional Activities table


In [ ]:
table2 = SpecialAppointments(
    doc=doc,worksheet='Special Appointments',
    table_name='Special Appointments',
    cumulative=True,
    ascending=False,
    template=professional_activities_template
)

Cleaning table for  Special Appointments
Initialized Special Appointments table


In [ ]:
#| export
# Function to extract start and end years
def extract_years(year_str):
    year_str = str(year_str)
    # Case 1 & 3: Single year (float) or non-aggregated years
    if re.match(r'^\d+(\.\d+)?$', year_str):
        year = int(float(year_str))
        return pd.Series([year, year])
    # Case 2: Pre-aggregated year range
    elif ' - ' in year_str:
        years = year_str.split(' - ')
        start_year = int(float(years[0]))
        end_year = int(float(years[-1]))
        return pd.Series([start_year, end_year])
    else:
        return pd.Series([None, None])

def combine_years(df):
    # Apply the function to create start and end year columns
    df[['StartYear', 'EndYear']] = df['Year'].apply(extract_years)

    # Remove rows where year extraction failed
    df = df.dropna(subset=['StartYear', 'EndYear'])

    # Ensure StartYear and EndYear are integers
    df['StartYear'] = df['StartYear'].astype(int)
    df['EndYear'] = df['EndYear'].astype(int)

    # Sort the dataframe
    df = df.sort_values(['Role', 'Organization', 'StartYear', 'EndYear'])

    # Group by Role and Organization
    grouped = df.groupby(['Role', 'Organization'])

    # Aggregate the data
    result = grouped.agg({
        'StartYear': 'min',
        'EndYear': 'max',
        'Date': 'first',
        'Eval': 'first',
        'Count': 'sum'
    }).reset_index()

    # Create the Year column with ranges
    result['Year'] = result.apply(lambda row: f"{row['StartYear']} - {row['EndYear']}" 
                                if row['StartYear'] != row['EndYear'] 
                                else str(row['StartYear']), axis=1)

    # Sort the result by the end year of activity
    result = result.sort_values('EndYear', ascending=False).drop(['StartYear', 'EndYear'], axis=1)

    return result

In [ ]:
#| export
def merge_tables(table1, table2):
    df = pd.concat([table1.df, table2.df])
    df = combine_years(df)
    df = df.sort_values(by=['Year', 'Role'], ascending=[True, False]).reset_index(drop=True)
    table1.df = df
    return table1

In [ ]:
# Add the df from table2 to table1
combined_activites = merge_tables(table1, table2)

/var/folders/bs/x9tn9jz91cv6hb3q6p4djbmw0000gn/T/ipykernel_9968/3054063915.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['StartYear'] = df['StartYear'].astype(int)
/var/folders/bs/x9tn9jz91cv6hb3q6p4djbmw0000gn/T/ipykernel_9968/3054063915.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['EndYear'] = df['EndYear'].astype(int)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()